In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
assert torch.cuda.is_available(), "Should use GPU-enabled colab"

device = torch.device('cuda:0')  

In [2]:
channels = {'P' : 0, 'N': 1, 'B' : 2, 'R' : 3, 'K' : 4,
            'Q' : 5, 'p' : 6, 'n' : 7, 'b' : 8, 'r' : 9,
            'k' : 10, 'q': 11, 'turn' : 12, 'Kc' : 13,
           'Qc' : 14, 'kc' : 15, 'qc' : 16,'en_passant' : 17}

def fen_to_tensor(fen):
# Encoding a chess position as an 18*8*8 tensor
# Takes a fen string --> returns a tensor representation
    fields = fen.split(' ')
    board_part, turn, castling, en_passant = fields[:4]
    tens = torch.zeros(18, 8, 8, dtype=torch.float32) 
    rows = board_part.split('/')
    for row_num, curr_row in enumerate(rows):
        col_num = 0
        for char in curr_row:
            if char in channels:
                tens[channels[char], row_num, col_num] = 1
                col_num += 1
            else:
                col_num += int(char)
    if turn == 'w':
        tens[12, :, :] = 1
    if castling != '-':
        for char in castling:
            channel = char + 'c'
            if channel in channels: # Safety check
                tens[channels[channel], :, :] = 1
    if en_passant != '-':
        row_num = 8 - int(en_passant[1])
        col_num = ord(en_passant[0]) - ord('a')
        tens[17, row_num, col_num] = 1
    return tens

In [3]:
class ChessDataset(Dataset):
    def __init__(self, fens, evaluations):
        """
        fens: list or pandas Series of FEN strings
        evaluations: list or pandas Series of float target values
        """
        self.fens = fens
        self.evaluations = evaluations

    def __len__(self):
        return len(self.fens)

    def __getitem__(self, idx):
        fen = self.fens[idx]
        target = self.evaluations[idx]
        
        # Convert inputs and targets to tensors
        x = fen_to_tensor(fen)
        y = torch.tensor([target], dtype=torch.float32)
        
        return x, y


class ChessMLP(nn.Module):
    def __init__(self, hidden_size=256):
        super().__init__()
        # 18 channels * 8 rows * 8 cols = 1152 features
        self.flatten = nn.Flatten()
        
        # Single hidden layer architecture
        self.network = nn.Sequential(
            nn.Linear(1152, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)


In [4]:
import pandas as pd

df = pd.read_parquet("Data/chessData_cleaned.parquet")

In [5]:
fens_list = df['FEN'].tolist()
evals_list = df['Target'].tolist()

dataset = ChessDataset(fens_list, evals_list)


In [6]:
total_size = len(dataset)
val_size = 500_000
test_size = 500_000
train_size = total_size - val_size - test_size

# 2. Split the dataset randomly
train_dataset, val_dataset, test_dataset = random_split(
    dataset, 
    [train_size, val_size, test_size]
)

# 3. Create separate DataLoaders
# Training loader needs shuffling to maintain stochastic gradient descent
train_loader = DataLoader(
    train_dataset, 
    batch_size=2048, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

# Validation and Test loaders do not need shuffling. 
# You can use a larger batch size here since no gradients are stored, saving VRAM.
val_loader = DataLoader(
    val_dataset, 
    batch_size=4096, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=4096, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

In [ ]:
model = ChessMLP(hidden_size=512).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()  # Sets layers like dropout/batchnorm to training mode
    running_train_loss = 0.0
    
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 20)
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move tensors to the GPU
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        # 1. Clear old gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        outputs = model(inputs)
        
        # 3. Compute loss
        loss = criterion(outputs, targets)
        
        # 4. Backward pass (compute new gradients)
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        running_train_loss += loss.item() * inputs.size(0)
        
        # Print progress every 1000 batches
        if (batch_idx + 1) % print_interval == 0 or (batch_idx + 1) == total_batches:
            percent_complete = 100.0 * (batch_idx + 1) / total_batches
            print(f"[{percent_complete:>3.0f}%] Batch {batch_idx+1}/{total_batches} - Loss: {loss.item():.4f}")

    avg_train_loss = running_train_loss / len(train_loader.dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()  # Sets layers to evaluation mode
    running_val_loss = 0.0
    
    # Disable gradient tracking for validation to save memory and speed up computation
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_val_loss += loss.item() * inputs.size(0)
            
    avg_val_loss = running_val_loss / len(val_loader.dataset)
    
    print(f"Epoch {epoch+1} Summary:")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- CHECKPOINTING ---
    # Save the model if it performs better on the validation set than previous epochs
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_chess_model.pth")
        print("--> Validation loss improved! Saved model checkpoint.")

print("\nTraining complete.")


Epoch 1/10
--------------------
